# Step 6: Locate TCR clonotypes spatially

This step performs clonotype-aware spatial analysis

## Setup and imports

In [ ]:
# Imports
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import squidpy as sq
import pandas as pd
import numpy as np
from pathlib import Path
import anndata as ad
import glob
import os
import shutil
import matplotlib.cm as cm
import scipy.stats
from scipy.stats import wilcoxon, zscore, gmean, pearsonr, norm, mannwhitneyu
from scipy import sparse
from statsmodels.stats.multitest import multipletests, fdrcorrection
from sklearn.neighbors import NearestNeighbors
from joblib import Parallel, delayed
from matplotlib.patches import Patch


In [ ]:
# Reproducibility settings
sc.settings.verbosity = 3
np.random.seed(26)

In [ ]:
# Path config
indir = '/path/to/integrated/processed_data/'
outdir = '/path/to/integrated/out/'
figdir = '/path/to/integrated/figures/'

In [ ]:
# Figure settings
plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams["font.family"] = "Helvetica"
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

## Load data

In [ ]:
adata = sc.read_h5ad(indir + 'xenium_integrated_labeled.h5ad')

## Define clones

Clones are chosen based on the following criteria:
- The clonotype chain (⍺β vs. γδ) matches the gene expression (CD4/CD8A vs. TRGC2)
- The clonotype passed the TCR probe quality dual criteria
- The clonotype generated statistically significant DE genes

In [ ]:
# all clones that passed the above criteria
pairs_map = {
    'γ Clone 2': ('clone2_', ['Patient2_DX', 'Patient2_PT', 'Patient5_DX'], 'γδ'),
    'γ Clone 4': ('clone4_', ['Patient2_PT'], 'γδ'),
    'α Clone 7': ('clone7_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'γ Clone 15': ('clone15_', ['Patient2_PT'], 'γδ'),
    'γ Clone 17': ('clone17_', ['Patient4_DX', 'Patient1_DX', 'Patient1_PT'], 'γδ'),
    'γ Clone 21': ('clone21_', ['Patient2_DX'], 'γδ'),
    'γ Clone 22': ('clone22_', ['Patient2_DX', 'Patient2_PT'], 'γδ'),
    'δ Clone 28': ('clone28_', ['Patient2_PT'], 'γδ'),
    'δ Clone 33': ('clone33_', ['Patient2_DX', 'Patient2_PT'], 'γδ'),
    'δ Clone 35': ('clone35_', ['Patient2_PT'], 'γδ'),
    'α Clone 39': ('clone39_', ['Patient2_PT'], '⍺β CD4'),
    'α Clone 43': ('clone43_', ['Patient2_PT'], '⍺β CD8'),
    'α Clone 55': ('clone55_', ['Patient5_DX'], '⍺β CD8'),
    'β Clone 65': ('clone65_', ['Patient2_PT'], '⍺β CD8'),
    'β Clone 82': ('clone82_', ['Patient5_PT'], '⍺β CD8'),
    'β Clone 84': ('clone84_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 1': ('clone45_', 'clone64_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 2': ('clone52_', 'clone60_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 3': ('clone51_', 'clone62_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 4': ('clone57_', 'clone87_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
}

In [ ]:
pairs_map = {
    'αβ Pair 1': ('clone45_', 'clone64_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 2': ('clone52_', 'clone60_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 3': ('clone51_', 'clone62_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 4': ('clone57_', 'clone87_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
}

In [ ]:
# all clones that passed the above criteria
pairs_map = {
    'γ Clone 2': ('clone2_', ['Patient2_DX', 'Patient2_PT', 'Patient5_DX'], 'γδ'),
    'γ Clone 17': ('clone17_', ['Patient4_DX', 'Patient1_DX', 'Patient1_PT'], 'γδ'),
    'αβ Pair 1': ('clone45_', 'clone64_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 2': ('clone52_', 'clone60_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 3': ('clone51_', 'clone62_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 4': ('clone57_', 'clone87_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    }

## Run DE on individual clones

In [ ]:
# Helper function
def get_clone_sum_initial(ad, prefix):
    genes = [g for g in ad.var_names if g.startswith(prefix)]
    if not genes:
        return np.zeros(ad.n_obs)
    X = ad[:, genes].X
    return X.sum(axis=1).A1 if scipy.sparse.issparse(X) else X.sum(axis=1)

In [ ]:
for sample_id in target_samples:
    print(f"\n{'='*25} Analyzing {sample_id} {'='*25}")
    
    sub_adata = adata[(adata.obs['sample'] == sample_id) & (adata.obs['celltype'] == "T")].copy()
    
    if sub_adata.n_obs == 0:
        print("No target cells found.")
        continue

    # Create DE object (exclude clone genes)
    non_clone_genes = [g for g in sub_adata.var_names if not g.startswith('clone')]
    adata_de = sub_adata[:, non_clone_genes].copy()
    
    # Build clone mask (1 or 2 clone prefixes)
    if len(clone_targets) == 1:
        c1 = get_clone_sum_initial(sub_adata, clone_targets[0]) > 0
        clone_mask = c1
        clone_label = clone_targets[0]
    elif len(clone_targets) == 2:
        c1 = get_clone_sum_initial(sub_adata, clone_targets[0]) > 0
        c2 = get_clone_sum_initial(sub_adata, clone_targets[1]) > 0
        clone_mask = c1 & c2
        clone_label = f"{clone_targets[0]} & {clone_targets[1]}"
    else:
        raise ValueError("clone_targets must have length 1 or 2.")

    # Final double-positive definition
    is_dp = clone_mask & is_target
    n_dp = int(is_dp.sum())
    
    if n_dp < 3:
        label_txt = f"{clone_label}" if gene_label is None else f"{clone_label} & {gene_label}"
        print(f"\n--- {label_txt} ---")
        print(f"Skipping: Only {n_dp} double-positive cells found.")
        continue
        
    # Label Groups: 'DoublePos' vs 'Other'
    adata_de.obs['comparison_group'] = 'Other'
    adata_de.obs.loc[is_dp, 'comparison_group'] = 'DoublePos'
    
    # Run DE Test
    try:
        sc.tl.rank_genes_groups(
            adata_de, 
            groupby='comparison_group', 
            groups=['DoublePos'], 
            reference='Other', 
            method='wilcoxon'
        )
        
        top_genes = sc.get.rank_genes_groups_df(adata_de, group='DoublePos').head(5)
        
        label_txt = f"{clone_label}" if gene_label is None else f"{clone_label} & {gene_label}"
        print(f"\n--- {label_txt} (n={n_dp}) ---")
        for i, row in top_genes.iterrows():
            print(f"{i+1}. {row['names']} (p-adj: {row['pvals_adj']:.2e}, logFC: {row['logfoldchanges']:.2f})")
                
    except Exception as e:
        print(f"Error analyzing {clone_label}: {e}")

## Visualize individual clones

In [ ]:
# Select individual clone for plotting
clone_name = "γ Clone 17" # clone of interest
gene_label = "TRGC2" 
value = pairs_map[clone_name]

if len(value) == 3:
    clone_targets = [value[0]]
    target_samples = value[1]
    tcr_type = value[2]
else:
    clone_targets = [value[0], value[1]]
    target_samples = value[2]
    tcr_type = value[3]

In [ ]:
def get_clone_sum(ad, prefix, clone_class=None):
    genes = [g for g in ad.var_names if g.startswith(prefix)]
    if not genes:
        return np.zeros(ad.n_obs)

    X = ad[:, genes].X
    if scipy.sparse.issparse(X):
        clone_sum = np.asarray(X.sum(axis=1)).ravel()
    else:
        clone_sum = np.asarray(X.sum(axis=1)).ravel()

    required_markers = {
        "γδ": ["TRGC2", "CD3E"],
        "⍺β CD8": ["CD8A", "CD3E"],
        "⍺β CD4": ["CD4", "CD3E"],
    }

    markers = required_markers.get(clone_class, [])
    if not markers:
        return clone_sum

    marker_mask = np.ones(ad.n_obs, dtype=bool)
    for m in markers:
        if m not in ad.var_names:
            return np.zeros(ad.n_obs)

        mx = ad[:, [m]].X  # keep 2D slice
        if scipy.sparse.issparse(mx):
            mvec = np.asarray(mx.toarray()).ravel()
        else:
            mvec = np.asarray(mx).ravel()

        marker_mask &= (mvec > 0)

    return np.where(marker_mask, clone_sum, 0.0)

In [ ]:
# Figure settings
n = len(target_samples)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
axes = np.atleast_1d(axes).flatten()

# Loop through samples and plot
for i, sample in enumerate(target_samples):
    ax = axes[i]
    
    sub = adata[adata.obs['sample'] == sample].copy()
    if sub.n_obs == 0:
        ax.axis('off'); continue

    # Target cell mask
    is_target = sub.obs['celltype'] == "T"

    if len(clone_targets) == 1:
        c1 = get_clone_sum(sub, clone_targets[0], tcr_type) > 0
        clone_mask = c1
        clone_label = clone_targets[0]
    elif len(clone_targets) == 2:
        c1 = get_clone_sum(sub, clone_targets[0], tcr_type) > 0
        c2 = get_clone_sum(sub, clone_targets[1], tcr_type) > 0
        clone_mask = c1 & c2
        clone_label = f"{clone_targets[0]} & {clone_targets[1]}"
    else:
        raise ValueError("clone_targets must have length 1 or 2.")

    # Final double-positive definition
    is_dp = clone_mask & is_target

    # Coordinates
    x = sub.obsm['spatial'][:, 0]
    y = sub.obsm['spatial'][:, 1]

    # Background (non-target)
    ax.scatter(x[~is_target], y[~is_target],
               c='#d3d3d3', s=2, edgecolors='none', alpha=0.8)

    # Target cells
    ax.scatter(x[is_target], y[is_target],
               c='#74add1', s=2, edgecolors='none', alpha=0.8)

    # Double positives
    ax.scatter(x[is_dp], y[is_dp],
               c='#000000', s=15, edgecolors='none', alpha=1)

    # Title
    if gene_label:
        title_txt = f"{sample}\n{clone_label} & {gene_label}+ (n={sum(is_dp)})"
    else:
        title_txt = f"{sample}\n{clone_label}+ (n={sum(is_dp)})"

    ax.set_title(title_txt, fontweight='bold')
    ax.axis('equal')
    ax.axis('off')

plt.tight_layout()

plt.savefig(f'{figdir}spatial_{clone_name}.png', 
    format='png', 
    transparent=True, 
    dpi=600,
    bbox_inches='tight')

plt.show()

## Quantify all clones/pairs that passed QC

In [ ]:
sample_order = [
    'Patient1_DX',
    'Patient1_PT',
    'Patient2_DX',
    'Patient2_PT',
    'Patient3_DX',
    'Patient3_PT',
    'Patient4_DX',
    'Patient4_PT',
    'Patient5_DX',
    'Patient5_PT'
]

In [ ]:
# Create heatmap
clone_labels = list(pairs_map.keys())

samples = sample_order.copy()

prop_df = pd.DataFrame(index=samples, columns=clone_labels, dtype=float)

for sample_id in samples:
    sub = adata[(adata.obs["sample"] == sample_id) & (adata.obs["celltype"] == "T")].copy()
    n_t = sub.n_obs

    if n_t == 0:
        continue

    for label, vals in pairs_map.items():
        if isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
            # (clone1, clone2, [samples], class)
            clone_targets = [vals[0], vals[1]]
            allowed_samples = vals[2]
            clone_class = vals[3]
        elif isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
            # (clone, [samples], class)
            clone_targets = [vals[0]]
            allowed_samples = vals[1]
            clone_class = vals[2]
        else:
            prop_df.loc[sample_id, label] = np.nan
            continue

        if sample_id not in allowed_samples:
            prop_df.loc[sample_id, label] = np.nan
            continue

        if len(clone_targets) == 1:
            pos_mask = get_clone_sum(sub, clone_targets[0], clone_class) > 0
        else:
            c1 = get_clone_sum(sub, clone_targets[0], clone_class) > 0
            c2 = get_clone_sum(sub, clone_targets[1], clone_class) > 0
            pos_mask = c1 & c2

        prop_df.loc[sample_id, label] = pos_mask.sum() / n_t

In [ ]:
keep_mask = prop_df.fillna(0).sum(axis=1) > 0
prop_df = prop_df.loc[keep_mask]
samples = prop_df.index.tolist()

annot_df = pd.DataFrame(index=prop_df.index, columns=prop_df.columns, dtype=object)

for sample_id in samples:
    sub = adata[(adata.obs["sample"] == sample_id) & (adata.obs["celltype"] == "T")]
    n_t = sub.n_obs
    if n_t == 0:
        annot_df.loc[sample_id, :] = ""
        continue

    for label, vals in pairs_map.items():
        val = prop_df.loc[sample_id, label]
        if pd.isna(val):
            annot_df.loc[sample_id, label] = ""
            continue

        count = int(round(val * n_t))
        annot_df.loc[sample_id, label] = f"{count}\n({val:.3f})"

In [ ]:
# Plot
plt.figure(figsize=(8, 3))
ax = sns.heatmap(
    prop_df,
    cmap="YlGnBu",
    vmin=0,
    vmax=np.nanmax(prop_df.values) if np.isfinite(np.nanmax(prop_df.values)) else 1,
    linewidths=0.5,
    linecolor="white",
    annot=annot_df,
    fmt="",
    cbar_kws={"label": "Clone count (proportion of T cells)"},
    mask=prop_df.isna()
)

ax.set_xlabel("Clone / Pair")
ax.set_ylabel("Sample")
plt.xticks(rotation=90, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    f"{figdir}all_clone_heatmap.pdf",
    bbox_inches="tight",
    transparent=True
)

plt.show()

In [ ]:
# Save source data
n_t_by_sample = (
    adata.obs.loc[adata.obs["celltype"] == "T", "sample"]
    .value_counts()
    .reindex(samples)
    .fillna(0)
    .astype(int)
)

count_df = prop_df.mul(n_t_by_sample, axis=0).round().astype("Int64")

source_long = (
    prop_df.stack(dropna=False)
    .rename("fraction")
    .reset_index()
    .rename(columns={"level_0": "sample", "level_1": "clone_pair"})
)
source_long["n_t_cells"] = source_long["sample"].map(n_t_by_sample)
source_long["count_positive"] = (
    source_long["fraction"] * source_long["n_t_cells"]
).round().astype("Int64")

source_long.to_csv(f"{figdir}all_clone_heatmap_source_data_long.csv", index=False)

## Evaluate 4 pairs again neoTCR gene lists

In [ ]:
# Define signatures, pairs, and samples
signatures = {
    'Caushi': ['HAVCR2', 'ITGAE', 'ENTPD1', 'PDCD1', 'CTLA4', 'TOX2', 'ZNF683', 'GNLY', 'BATF', 'CXCL13'],
    'Oliveira': ['KRT86', 'RDH10', 'TYMS', 'HMOX1', 'GNG4', 'CXCL13', 'AFAP1L2', 'ACP5', 'MYO1E', 'LAYN', 'TNS3', 'TNFSF4', 'AKAP5', 'HAVCR2', 'ENTPD1', 'SLC2A8', 'ZBED2', 'MCM5', 'CAV1', 'GOLIM4', 'VCAM1', 'PON2', 'MTSS1', 'CD38', 'MS4A6A', 'TOX2', 'CSF1', 'GALNT2', 'FXYD2', 'PLPP1', 'LMCD1', 'MYL6B', 'LAG3', 'HLA-DRA', 'IGFLR1', 'CCDC50', 'CD27', 'KIAA1324', 'CDKN2A', 'CD70', 'ABHD6', 'CTLA4', 'PDCD1', 'GEM', 'NUSAP1', 'TOX', 'CXCR6', 'NMB', 'HOPX', 'CLIC3', 'INPP5F', 'SNAP47', 'TSHZ2', 'HLA-DMA', 'SIT1', 'HLA-DRB1', 'TUBB', 'PYCARD', 'ADGRG1', 'HLA-DQA1', 'PRF1', 'HLA-DPA1', 'PTMS', 'CKS1B', 'HIPK2', 'CHST12', 'LSP1', 'FAM3C', 'SLC1A4', 'NUDT1', 'DNPH1'],
    'Lowery': ['ATP10D', 'GZMB', 'ENTPD1', 'KIR2DL4', 'LAYN', 'HTRA1', 'CD70', 'CXCR6', 'HMOX1', 'ADGRG1', 'LRRN3', 'ACP5', 'CTSW', 'GALNT2', 'CARS', 'LAG3', 'TOX', 'PTPRCAP', 'ASB2', 'ITGB7', 'PTMS', 'CD8A', 'GPR68', 'NSMCE1', 'ABI3', 'SLC1A4', 'PLEKHF1', 'CD8B', 'CCL4', 'NKG7', 'CLIC3', 'NDFIP2', 'PLPP1', 'PCED1B', 'CXCL13', 'PDCD1', 'PRF1', 'HLA-DMA'],
    'Hanada': ['ENTPD1', 'CXCL13', 'HMOX1', 'PDCD1', 'LAYN', 'CD27', 'HAVCR2', 'TNFRSF9', 'MIR155HG', 'BATF', 'TIGIT', 'GZMH', 'CD70', 'TMEM121', 'LRRN3', 'NHS', 'TTN', 'ASB2', 'SIRPG', 'ANKS1B']
}

pairs = [
    ('clone45_', 'clone64_'),
    ('clone52_', 'clone60_'),
    ('clone51_', 'clone62_'),
    ('clone57_', 'clone87_')
]

samples = ['Patient2_DX', 'Patient2_PT']

pair_color = "#FFA63B"
other_color = "#E0E0E0"

In [ ]:
# Calculate signature scores
for name, genes in signatures.items():
    valid_genes = [g for g in genes if g in adata.var_names]
    if valid_genes:
        sc.tl.score_genes(adata, gene_list=valid_genes, score_name=f'score_{name}')
    else:
        print(f"Warning: No valid genes found for signature {name}")

In [ ]:
# Helper to identify cells
def get_clone_mask(ad, c1, c2):
    g1 = [g for g in ad.var_names if g.startswith(c1)]
    g2 = [g for g in ad.var_names if g.startswith(c2)]
    if not g1 or not g2:
        return np.zeros(ad.n_obs, dtype=bool)

    e1 = ad[:, g1].X.sum(axis=1)
    e2 = ad[:, g2].X.sum(axis=1)
    if hasattr(e1, 'A1'): e1 = e1.A1
    if hasattr(e2, 'A1'): e2 = e2.A1

    return (e1 > 0) & (e2 > 0)

In [ ]:
# Collect all tests for FDR
results_data = []
for sample in samples:
    sub = adata[(adata.obs['sample'] == sample) & (adata.obs['celltype'] == 'T')].copy()

    for (c1, c2) in pairs:
        is_pair = get_clone_mask(sub, c1, c2)
        n_pair = int(is_pair.sum())
        n_other = int(len(is_pair) - n_pair)

        if n_pair < 3 or n_other < 3:
            for sig_name in signatures.keys():
                results_data.append({
                    "sample": sample,
                    "pair": f"{c1}+{c2}",
                    "signature": sig_name,
                    "p": np.nan
                })
            continue

        for sig_name in signatures.keys():
            score_col = f"score_{sig_name}"
            s_pair = sub.obs.loc[is_pair, score_col]
            s_other = sub.obs.loc[~is_pair, score_col]

            stat, p = mannwhitneyu(s_pair, s_other, alternative='two-sided')
            results_data.append({
                "sample": sample,
                "pair": f"{c1}+{c2}",
                "signature": sig_name,
                "p": float(p)
            })

results_df = pd.DataFrame(results_data)

In [ ]:
# FDR across all valid p-values
valid = results_df["p"].notna()
reject, q = fdrcorrection(results_df.loc[valid, "p"].values, alpha=0.05)
results_df.loc[valid, "q"] = q
results_df.loc[valid, "significant_fdr"] = reject

def q_to_stars(qv):
    if pd.isna(qv):
        return ""
    if qv < 1e-3:
        return "***"
    if qv < 1e-2:
        return "**"
    if qv < 5e-2:
        return "*"
    return ""

results_df["stars"] = results_df["q"].apply(q_to_stars)

In [ ]:
# pair labels
pair_ids = [f"Pair{i+1}" for i in range(len(pairs))]
pair_label_map = {pair_ids[i]: f"{pairs[i][0]}+{pairs[i][1]}" for i in range(len(pairs))}
pair_to_id = {v: k for k, v in pair_label_map.items()}

In [ ]:
# Helper: sample -> timepoint
def sample_to_timepoint(s):
    s = str(s)
    if s.endswith("_DX"):
        return "DX"
    if s.endswith("_PT"):
        return "PT"
    return np.nan

In [ ]:
# Build long plot_df for boxplots
rows = []
for sample in samples:
    sub = adata[(adata.obs["sample"] == sample) & (adata.obs["celltype"] == "T")].copy()
    if sub.n_obs == 0:
        continue

    tp = sample_to_timepoint(sample)

    for (c1, c2) in pairs:
        pair_label = f"{c1}+{c2}"
        pid = pair_to_id[pair_label]

        is_pair = get_clone_mask(sub, c1, c2)
        if is_pair.sum() < 3 or (~is_pair).sum() < 3:
            continue

        for sig in signatures.keys():
            score_col = f"score_{sig}"
            if score_col not in sub.obs.columns:
                continue

            # Pair cells
            for v in sub.obs.loc[is_pair, score_col].values:
                rows.append({
                    "sample": sample,
                    "timepoint": tp,
                    "pair": pair_label,
                    "pair_id": pid,
                    "signature": sig,
                    "group": "Pair",
                    "score": float(v),
                })

            # Other cells
            for v in sub.obs.loc[~is_pair, score_col].values:
                rows.append({
                    "sample": sample,
                    "timepoint": tp,
                    "pair": pair_label,
                    "pair_id": pid,
                    "signature": sig,
                    "group": "Other",
                    "score": float(v),
                })

plot_df = pd.DataFrame(rows)

# Build tests_df expected by (1-83)
tests_df = results_df.copy()
tests_df["timepoint"] = tests_df["sample"].apply(sample_to_timepoint)
tests_df["pair_id"] = tests_df["pair"].map(pair_to_id)

In [ ]:
# Save test and plot data
tests_df.to_csv(f"{outdir}fig7f_neotcr_test_data.csv", index=True)

mid = (len(plot_df) + 1) // 2  # first half gets extra row if odd

plot_df.iloc[:mid].to_csv(f"{outdir}fig7f_neotcr_plot_data_pt1.csv", index=True)
plot_df.iloc[mid:].to_csv(f"{outdir}fig7f_neotcr_plot_data_pt2.csv", index=True)

In [ ]:
# Plot
FONT_SIZE = 6

sig_order = list(signatures.keys())
tp_present = plot_df["timepoint"].dropna().unique().tolist()
tp_order = [t for t in ["DX", "PT"] if t in tp_present] + [t for t in tp_present if t not in ["DX", "PT"]]

# Global y-axis limits across all panels
score_vals = plot_df["score"].dropna().values
if len(score_vals) == 0:
    y_lower, y_upper = 0.0, 1.0
else:
    y_lower = float(np.min(score_vals))
    y_upper = float(np.max(score_vals))
    pad = 0.05 * (y_upper - y_lower if y_upper > y_lower else 1.0)
    y_lower -= pad
    y_upper += pad

fig, axes = plt.subplots(
    nrows=len(tp_order),
    ncols=len(sig_order),
    figsize=(13/2.54, 5/2.54),
    squeeze=False
)

for r, tp in enumerate(tp_order):
    for c, sig in enumerate(sig_order):
        ax = axes[r, c]
        d = plot_df[(plot_df["timepoint"] == tp) & (plot_df["signature"] == sig)].copy()

        if d.empty:
            ax.axis("off")
            continue

        sns.boxplot(
            data=d,
            x="pair_id", y="score", hue="group",
            order=pair_ids,
            hue_order=["Pair", "Other"],
            palette={"Pair": pair_color, "Other": other_color},
            showfliers=False,
            ax=ax,
            linewidth=0.5,
            boxprops={"linewidth": 0.5},
            whiskerprops={"linewidth": 0.5},
            capprops={"linewidth": 0.5},
            medianprops={"linewidth": 0.5},
        )

        # Force same y-axis for all subplots
        ax.set_ylim(y_lower, y_upper)

        # Star position based on fixed global scale
        y_text = y_upper - 0.03 * (y_upper - y_lower)

        for i_pid, pid in enumerate(pair_ids):
            row = tests_df[
                (tests_df["timepoint"] == tp) &
                (tests_df["signature"] == sig) &
                (tests_df["pair_id"] == pid)
            ]
            if len(row) == 1:
                star = row["stars"].iloc[0]
                if star:
                    ax.text(
                        i_pid, y_text, star,
                        ha="center", va="top",
                        fontsize=FONT_SIZE, fontweight="bold"
                    )

        if r == 0:
            ax.set_xticklabels([])
            ax.set_xlabel("")
        else:
            ax.set_xticklabels(pair_ids, rotation=0, fontsize=FONT_SIZE)
            ax.set_xlabel("")

        if r == len(tp_order) - 1:
            ax.set_title("")
        else:
            ax.set_title(f"{sig}", fontsize=FONT_SIZE)

        if c == 0:
            ax.set_ylabel("Score", fontsize=FONT_SIZE)
        else:
            ax.set_ylabel("")

        ax.tick_params(axis="both", which="both", width=0.5, length=2, labelsize=FONT_SIZE)

        ax.grid(False)
        for side in ["top", "right", "left", "bottom"]:
            ax.spines[side].set_visible(False)

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

plt.tight_layout()
plt.savefig(
    f"{figdir}neotcr_2x4_timepoint_by_signature_boxes_by_pair.pdf",
    format="pdf",
    transparent=False,
    bbox_inches="tight",
)
plt.show()

## Clonotype vs. domain analysis

### Dotplot

In [ ]:
# Dotplot settings
immune_rich_clusters = ["0", "4", "8", "9"]
neuroblast_rich_clusters = ["1", "2", "6", "7", "10", "12", "14", "16", "17", "18"]

def classify_3_domains(n):
    n_str = str(n)
    if n_str in immune_rich_clusters:
        return "Immune-rich"
    elif n_str in neuroblast_rich_clusters:
        return "Neuroblast-rich"
    else:
        return "Other"

categories_order = ["Immune-rich", "Neuroblast-rich", "Other"]

genes_to_plot = [
    "TCF7", "LEF1", "SELL", "IL7R",
    "THEMIS", "FOS",
    "IFNG", "KLRD1", "GZMH", "GZMB",
    "TOX", "LAG3",
]

def get_single_clone_mask(ad, c1):
    g1 = [g for g in ad.var_names if g.startswith(c1)]
    if not g1:
        return np.zeros(ad.n_obs, dtype=bool)
    e1 = ad[:, g1].X.sum(axis=1)
    if hasattr(e1, "A1"):
        e1 = e1.A1
    else:
        e1 = np.asarray(e1).ravel()
    return e1 > 0

In [ ]:
for label, vals in pairs_map.items():
    if isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
        c1 = vals[0]
        target_samples = vals[1]
        tcr_type = vals[2]
        entry_type = "clone"
    elif isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
        c1, c2 = vals[0], vals[1]
        target_samples = vals[2]
        tcr_type = vals[3]
        entry_type = "pair"
    else:
        continue

    sub_all = adata[
        (adata.obs["sample"].isin(target_samples)) &
        (adata.obs["celltype"] == "T")
    ].copy()
    if sub_all.n_obs == 0:
        continue

    if entry_type == "clone":
        clone_mask = get_single_clone_mask(sub_all, c1)
    else:
        clone_mask = get_clone_mask(sub_all, c1, c2)

    sub_all = sub_all[clone_mask].copy()
    if sub_all.n_obs == 0:
        continue

    sub_all.obs["domain"] = sub_all.obs["neigh_kmeans"].map(classify_3_domains)
    sub_all = sub_all[sub_all.obs["domain"].notna()].copy()
    if sub_all.n_obs == 0:
        continue

    present = sub_all.obs["domain"].value_counts()
    if present.shape[0] < 2:
        continue

    categories_present = [c for c in categories_order if c in present.index]
    if len(categories_present) < 2:
        continue

    genes = [g for g in genes_to_plot if g in sub_all.var_names]
    if len(genes) == 0:
        continue

    # Dotplot in same style pattern as 2.subcluster_and_visualize_metadata
    dp = sc.pl.dotplot(
        sub_all,
        genes,
        groupby="domain",
        categories_order=categories_present,
        cmap="YlOrRd",
        return_fig=True,
        show=False,
        standard_scale="var",
    )

    dp = dp.style(
        smallest_dot=0.01,
        largest_dot=70,
    )

    ax_dict = dp.get_axes()

    # Make ticks thinner
    for _, ax in ax_dict.items():
        if ax is None:
            continue
        ax.tick_params(axis="both", which="both", width=0.5, length=2)

    # Main panel border
    main_ax = ax_dict.get("mainplot_ax", None)
    if main_ax is not None:
        for spine in main_ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_color("#000000")

    # Color legend bar border
    cax = ax_dict.get("color_legend_ax", None)
    if cax is not None:
        for spine in cax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_color("#000000")

    # Title + figure sizing/saving
    fig = dp.fig
    fig.suptitle(f"{label} ({tcr_type}): 3-way domains", fontsize=6, y=1.02)
    fig.set_size_inches(10/2.54, 3.5/2.54, forward=True)
    fig.tight_layout()
    fig.canvas.draw()

    display(fig)
    fig.savefig(f"{figdir}dotplot_{label}_domains.pdf", bbox_inches="tight")
    plt.close(fig)

In [ ]:
# Statistics
MIN_CELLS_PER_DOMAIN = 10      # per sample, per domain, for a comparison to be included
N_PERM = 10000
RNG_SEED = 42
domain_pairs = [
    ("Immune-rich", "Neuroblast-rich"),
    ("Immune-rich", "Other"),
    ("Neuroblast-rich", "Other"),
]

rng = np.random.default_rng(RNG_SEED)


# Helpers
def _to_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

def _get_entry_info(label, vals):
    # vals format:
    # clone: (c1, [samples], tcr_type)
    # pair : (c1, c2, [samples], tcr_type)
    if isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
        return {"entry_type": "clone", "c1": vals[0], "c2": None, "samples": vals[1], "tcr_type": vals[2]}
    if isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
        return {"entry_type": "pair", "c1": vals[0], "c2": vals[1], "samples": vals[2], "tcr_type": vals[3]}
    return None

def _obs_stat_unweighted_mean_of_sample_deltas(df, d1, d2):
    # df columns: sample, domain, expr
    sample_deltas = []
    for s, g in df.groupby("sample"):
        m1 = g.loc[g["domain"] == d1, "expr"].mean()
        m2 = g.loc[g["domain"] == d2, "expr"].mean()
        sample_deltas.append(m2 - m1)  # domain2 - domain1
    return float(np.mean(sample_deltas)), np.array(sample_deltas, dtype=float)

def _stratified_perm_pvalue(cell_df, d1, d2, n_perm=10000, rng=None):
    """
    cell_df columns: sample, domain, expr
    Permutes domain labels within each sample (restricted to d1/d2 cells),
    preserving sample structure and domain counts.
    """
    if rng is None:
        rng = np.random.default_rng(1)

    # observed
    obs_stat, sample_deltas = _obs_stat_unweighted_mean_of_sample_deltas(cell_df, d1, d2)

    # pre-split by sample for speed
    by_sample = []
    for s, g in cell_df.groupby("sample"):
        dom = g["domain"].to_numpy()
        expr = g["expr"].to_numpy()
        by_sample.append((s, dom, expr))

    perm_stats = np.empty(n_perm, dtype=float)

    for i in range(n_perm):
        deltas_i = []
        for _, dom, expr in by_sample:
            perm_dom = rng.permutation(dom)  # within-sample permutation
            m1 = expr[perm_dom == d1].mean()
            m2 = expr[perm_dom == d2].mean()
            deltas_i.append(m2 - m1)
        perm_stats[i] = np.mean(deltas_i)

    p_two_sided = (np.sum(np.abs(perm_stats) >= np.abs(obs_stat)) + 1.0) / (n_perm + 1.0)
    return obs_stat, p_two_sided, sample_deltas

# Main

count_rows = []
stat_rows = []

for label, vals in pairs_map.items():
    info = _get_entry_info(label, vals)
    if info is None:
        continue

    # subset T cells in target samples
    sub_all = adata[
        (adata.obs["sample"].isin(info["samples"])) &
        (adata.obs["celltype"] == "T")
    ].copy()
    if sub_all.n_obs == 0:
        continue

    # clone/pair mask
    if info["entry_type"] == "clone":
        mask = get_single_clone_mask(sub_all, info["c1"])
    else:
        mask = get_clone_mask(sub_all, info["c1"], info["c2"])

    sub = sub_all[mask].copy()
    if sub.n_obs == 0:
        continue

    # domains
    sub.obs["domain"] = sub.obs["neigh_kmeans"].map(classify_3_domains)
    sub = sub[sub.obs["domain"].isin(categories_order)].copy()
    if sub.n_obs == 0:
        continue

    # counts table (required for reviewer transparency)
    ctab = (
        sub.obs.groupby(["sample", "domain"])
        .size()
        .reset_index(name="n_clonepos_cells")
    )
    ctot = sub.obs.groupby("sample").size().reset_index(name="n_clonepos_total")
    ctab = ctab.merge(ctot, on="sample", how="left")
    ctab["prop_within_clonepos"] = ctab["n_clonepos_cells"] / ctab["n_clonepos_total"]
    ctab["label"] = label
    ctab["entry_type"] = info["entry_type"]
    ctab["tcr_type"] = info["tcr_type"]
    count_rows.append(ctab)

    # genes present
    genes = [g for g in genes_to_plot if g in sub.var_names]
    if len(genes) == 0:
        continue

    X = _to_dense(sub[:, genes].X)
    expr_df = pd.DataFrame(X, columns=genes, index=sub.obs_names)
    meta = sub.obs[["sample", "domain"]].copy()
    long_df = meta.join(expr_df).reset_index(drop=True).melt(
        id_vars=["sample", "domain"], var_name="gene", value_name="expr"
    )

    # pairwise domain stats per gene
    for g in genes:
        dg = long_df[long_df["gene"] == g].copy()

        for d1, d2 in domain_pairs:
            dsub = dg[dg["domain"].isin([d1, d2])].copy()
            if dsub.empty:
                continue

            # keep only samples with both domains and minimum cells per domain
            keep_samples = []
            for s, gs in dsub.groupby("sample"):
                n1 = int((gs["domain"] == d1).sum())
                n2 = int((gs["domain"] == d2).sum())
                if n1 >= MIN_CELLS_PER_DOMAIN and n2 >= MIN_CELLS_PER_DOMAIN:
                    keep_samples.append(s)

            dsub = dsub[dsub["sample"].isin(keep_samples)].copy()
            n_samples = dsub["sample"].nunique()

            row = {
                "label": label,
                "entry_type": info["entry_type"],
                "tcr_type": info["tcr_type"],
                "gene": g,
                "domain_1": d1,
                "domain_2": d2,
                "n_samples_with_both_domains": int(n_samples),
                "min_cells_per_domain_rule": int(MIN_CELLS_PER_DOMAIN),
                "obs_mean_delta_d2_minus_d1": np.nan,      # main effect size
                "median_sample_delta_d2_minus_d1": np.nan, # robust effect size
                "perm_p_two_sided": np.nan,
                "inference_tier": "not_tested"
            }

            if n_samples >= 2:
                obs_delta, p_perm, sample_deltas = _stratified_perm_pvalue(
                    dsub[["sample", "domain", "expr"]], d1, d2, n_perm=N_PERM, rng=rng
                )
                row["obs_mean_delta_d2_minus_d1"] = float(obs_delta)
                row["median_sample_delta_d2_minus_d1"] = float(np.median(sample_deltas))
                row["perm_p_two_sided"] = float(p_perm)
                row["inference_tier"] = "exploratory_n2" if n_samples == 2 else "sample_level"

            stat_rows.append(row)

counts_df = pd.concat(count_rows, ignore_index=True) if len(count_rows) else pd.DataFrame()
stats_df = pd.DataFrame(stat_rows)

# FDR corrections
if not stats_df.empty:
    stats_df["qval_bh_global"] = np.nan
    valid = stats_df["perm_p_two_sided"].notna()
    if valid.any():
        stats_df.loc[valid, "qval_bh_global"] = multipletests(
            stats_df.loc[valid, "perm_p_two_sided"].values, method="fdr_bh"
        )[1]

    stats_df["qval_bh_within_label"] = np.nan
    for lb, idx in stats_df.groupby("label").groups.items():
        idx = np.array(list(idx))
        m = stats_df.loc[idx, "perm_p_two_sided"].notna().values
        if m.any():
            stats_df.loc[idx[m], "qval_bh_within_label"] = multipletests(
                stats_df.loc[idx[m], "perm_p_two_sided"].values, method="fdr_bh"
            )[1]

    def q_to_stars(q):
        if pd.isna(q): return ""
        if q < 1e-3: return "***"
        if q < 1e-2: return "**"
        if q < 5e-2: return "*"
        return ""
    stats_df["stars_within_label"] = stats_df["qval_bh_within_label"].apply(q_to_stars)

# Save
if not counts_df.empty:
    counts_df.to_csv(f"{outdir}fig5_clonepair_domain_cell_counts.csv", index=False)
if not stats_df.empty:
    stats_df.to_csv(f"{outdir}fig5_clonepair_domain_expr_stratified_permutation_stats.csv", index=False)

display(counts_df.sort_values(["label", "sample", "domain"]) if not counts_df.empty else counts_df)
display(
    stats_df.sort_values(["label", "gene", "qval_bh_within_label"], na_position="last")
    if not stats_df.empty else stats_df
)

## Extra code chunks for additional visualization
Not included in the manuscript

#### Single-cell level clonotype vs. domain

In [ ]:
# Helper
def classify_3_domains(n):
    n_str = str(n)
    if n_str in immune_rich_clusters:
        return "Immune-rich"
    elif n_str in neuroblast_rich_clusters:
        return "Neuroblast-rich"
    else:
        return "Other"

In [ ]:
immune_rich_clusters = ["0", "4", "8", "9"]
neuroblast_rich_clusters = ["1", "2", "6", "7", "10", "12", "14", "16", "17", "18"]

categories_order = ["Immune-rich", "Neuroblast-rich", "Other"]

genes_to_plot = [
    "TCF7", "SELL", "CCR7",
    "IL7R", "KLRD1", "LAG3"
]

samples_to_plot = ["Patient2_DX", "Patient2_PT"]

In [ ]:
MIN_CELLS_TO_PLOT = 20
MIN_CELLS_PER_DOMAIN_TEST = 5

entry_items = []
for label, vals in pairs_map.items():
    if isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
        c1, target_samples, tcr_type = vals
        entry_items.append({
            "label": label,
            "entry_type": "clone",
            "c1": c1,
            "c2": None,
            "target_samples": target_samples,
            "tcr_type": tcr_type
        })
    elif isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
        c1, c2, target_samples, tcr_type = vals
        entry_items.append({
            "label": label,
            "entry_type": "pair",
            "c1": c1,
            "c2": c2,
            "target_samples": target_samples,
            "tcr_type": tcr_type
        })

if len(entry_items) == 0:
    raise ValueError("No valid clone/pair entries found in pairs_map.")

# Samples inferred from pairs_map
samples_from_map = []
for e in entry_items:
    for s in e["target_samples"]:
        if s not in samples_from_map:
            samples_from_map.append(s)

samples_to_plot = samples_from_map

# Helper for single clone mask
def get_single_clone_mask(ad, c1):
    g1 = [g for g in ad.var_names if g.startswith(c1)]
    if not g1:
        return np.zeros(ad.n_obs, dtype=bool)
    e1 = ad[:, g1].X.sum(axis=1)
    if hasattr(e1, "A1"):
        e1 = e1.A1
    else:
        e1 = np.asarray(e1).ravel()
    return e1 > 0

pair_sc_long_by_sample = {s: {} for s in samples_to_plot}
# sample -> label -> (sc_long, tcr_type, domains_present, domains_plot_order)

for sample_id in samples_to_plot:
    sub_all = adata[
        (adata.obs["sample"] == sample_id) &
        (adata.obs["celltype"] == "T")
    ].copy()
    if sub_all.n_obs == 0:
        continue

    for e in entry_items:
        label = e["label"]

        # evaluate only in allowed samples
        if sample_id not in e["target_samples"]:
            continue

        # clone-positive or pair-positive mask
        if e["entry_type"] == "clone":
            clone_mask = get_single_clone_mask(sub_all, e["c1"])
        else:
            clone_mask = get_clone_mask(sub_all, e["c1"], e["c2"])

        sub_entry = sub_all[clone_mask].copy()

        if sub_entry.n_obs < MIN_CELLS_TO_PLOT:
            continue

        sub_entry.obs["domain"] = sub_entry.obs["neigh_kmeans"].map(classify_3_domains)
        sub_entry = sub_entry[sub_entry.obs["domain"].notna()].copy()
        if sub_entry.n_obs < MIN_CELLS_TO_PLOT:
            continue

        present = sub_entry.obs["domain"].value_counts()
        domains_present = [c for c in categories_order if c in present.index]
        if len(domains_present) < 2:
            continue

        # genes present
        genes_present = [g for g in genes_to_plot if g in sub_entry.var_names]
        if len(genes_present) == 0:
            continue

        X = sub_entry[:, genes_present].X
        if hasattr(X, "toarray"):
            X = X.toarray()
        else:
            X = np.asarray(X)

        expr_df = pd.DataFrame(X, columns=genes_present, index=sub_entry.obs_names)

        meta_df = sub_entry.obs[["domain"]].copy()
        meta_df["domain"] = pd.Categorical(meta_df["domain"], categories=categories_order, ordered=True)

        sc_df = pd.concat([meta_df, expr_df], axis=1).reset_index(names="cell_id")
        sc_long = sc_df.melt(
            id_vars=["cell_id", "domain"],
            var_name="gene",
            value_name="expr"
        )
        sc_long["sample"] = sample_id
        sc_long["pair"] = label  # keeps compatibility with downstream code

        pair_sc_long_by_sample[sample_id][label] = (
            sc_long, e["tcr_type"], domains_present, categories_order
        )

comp_order = [
    ("Immune-rich", "Neuroblast-rich"),
    ("Immune-rich", "Other"),
    ("Neuroblast-rich", "Other"),
]

pairwise_rows = []

for sample_id in samples_to_plot:
    for label, payload in pair_sc_long_by_sample[sample_id].items():
        sc_long, tcr_type, domains_present, _ = payload
        present_set = set(domains_present)

        for gene in genes_to_plot:
            d_gene = sc_long[sc_long["gene"] == gene].copy()
            if d_gene.empty:
                continue

            for slot_idx, (d1, d2) in enumerate(comp_order, start=1):
                row = {
                    "sample": sample_id,
                    "pair": label,
                    "gene": gene,
                    "tcr_type": tcr_type,
                    "slot": slot_idx,
                    "domain_1": d1,
                    "domain_2": d2,
                    "n_1": np.nan,
                    "n_2": np.nan,
                    "median_1": np.nan,
                    "median_2": np.nan,
                    "median_diff_1_minus_2": np.nan,
                    "mean_diff_1_minus_2": np.nan,
                    "u_stat": np.nan,
                    "pval": np.nan,
                    "cliffs_delta": np.nan,
                    "common_language": np.nan,
                    "valid_test": False
                }

                if d1 not in present_set or d2 not in present_set:
                    pairwise_rows.append(row)
                    continue

                x1 = d_gene.loc[d_gene["domain"] == d1, "expr"].dropna().values
                x2 = d_gene.loc[d_gene["domain"] == d2, "expr"].dropna().values

                n1, n2 = len(x1), len(x2)
                row["n_1"] = n1
                row["n_2"] = n2

                # NEW: require enough cells per compared domain
                if n1 < MIN_CELLS_PER_DOMAIN_TEST or n2 < MIN_CELLS_PER_DOMAIN_TEST:
                    pairwise_rows.append(row)
                    continue

                row["median_1"] = float(np.median(x1))
                row["median_2"] = float(np.median(x2))
                row["median_diff_1_minus_2"] = float(np.median(x1) - np.median(x2))
                row["mean_diff_1_minus_2"] = float(np.mean(x1) - np.mean(x2))

                try:
                    mwu = mannwhitneyu(x1, x2, alternative="two-sided")
                    u = float(mwu.statistic)
                    pval = float(mwu.pvalue)
                    delta = (2.0 * u) / (n1 * n2) - 1.0
                    cl = u / (n1 * n2)

                    row["u_stat"] = u
                    row["pval"] = pval
                    row["cliffs_delta"] = float(delta)
                    row["common_language"] = float(cl)
                    row["valid_test"] = True
                except Exception:
                    pass

                pairwise_rows.append(row)

pairwise_stats_df = pd.DataFrame(pairwise_rows)

# BH-FDR across all valid tests
pairwise_stats_df["qval"] = np.nan
valid = pairwise_stats_df["valid_test"] & pairwise_stats_df["pval"].notna()
if valid.any():
    pairwise_stats_df.loc[valid, "qval"] = multipletests(
        pairwise_stats_df.loc[valid, "pval"].values,
        method="fdr_bh"
    )[1]

def format_sig_effect(q, delta):
    if pd.isna(q) or pd.isna(delta):
        return ""
    sig = "*" if q < 0.05 else "ns"
    return f"{sig} (δ={delta:+.2f})"

def add_sig_line(ax, x1, x2, y, label, lw=0.8):
    ax.hlines(y=y, xmin=x1, xmax=x2, colors="black", linewidth=lw, clip_on=False)
    ax.text((x1 + x2) / 2, y - 0.001, label, ha="center", va="bottom", fontsize=6)

def get_rank_palette(d_gene, domains_plot_order):
    low_col = "#edf8b1"
    mid_col = "#7fcdbb"
    high_col = "#2c7fb8"

    centers = d_gene.groupby("domain")["expr"].mean().reindex(domains_plot_order)
    ranked = centers.dropna().sort_values().index.tolist()

    color_map = {dom: "#cccccc" for dom in domains_plot_order}
    if len(ranked) == 2:
        color_map[ranked[0]] = low_col
        color_map[ranked[1]] = high_col
    elif len(ranked) >= 3:
        color_map[ranked[0]] = low_col
        color_map[ranked[-1]] = high_col
        for dom in ranked[1:-1]:
            color_map[dom] = mid_col

    return [color_map[dom] for dom in domains_plot_order]

# ---- Plot one figure per sample ----
plot_labels = [e["label"] for e in entry_items]  # includes all clones + pairs
n_rows = len(plot_labels)
n_cols = len(genes_to_plot)

for sample_id in samples_to_plot:
    fig_height = n_rows * 2.5
    fig_width = max(8, n_cols * 2.0)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(fig_width, fig_height),
        squeeze=False
    )

    for r, label in enumerate(plot_labels):
        payload = pair_sc_long_by_sample[sample_id].get(label, None)
        if payload is None:
            for c in range(n_cols):
                axes[r, c].axis("off")
            continue

        sc_long, tcr_type, domains_present, domains_plot_order = payload

        for c, gene in enumerate(genes_to_plot):
            ax = axes[r, c]
            d_gene = sc_long[sc_long["gene"] == gene].copy()
            if d_gene.empty:
                ax.axis("off")
                continue

            panel_palette = get_rank_palette(d_gene, domains_plot_order)

            sns.violinplot(
                data=d_gene,
                x="domain", y="expr",
                order=domains_plot_order,
                ax=ax,
                inner=None,
                palette=panel_palette,
                linewidth=1
            )

            ax.set_ylim(0, 10.5)

            if r == 0:
                ax.set_title(gene)
            else:
                ax.set_title("")

            if r == n_rows - 1:
                ax.tick_params(axis="x", labelrotation=90)
                for tick in ax.get_xticklabels():
                    tick.set_ha("right")
            else:
                ax.set_xticklabels([])
                ax.set_xlabel("")

            if c == 0:
                n_cells = int(d_gene["cell_id"].nunique())
                ax.set_ylabel(f"{label}\n({tcr_type})\n(n={n_cells})")
            else:
                ax.set_ylabel("")

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.spines["left"].set_visible(True)
            ax.spines["bottom"].set_visible(True)

            sub_stats = pairwise_stats_df[
                (pairwise_stats_df["sample"] == sample_id) &
                (pairwise_stats_df["pair"] == label) &
                (pairwise_stats_df["gene"] == gene)
            ].copy()

            xpos = {dom: i for i, dom in enumerate(domains_plot_order)}
            base = 7.5
            step = 0.9
            drawn = 0

            for slot_idx, (d1, d2) in enumerate(comp_order, start=1):
                rr = sub_stats[sub_stats["slot"] == slot_idx]
                if rr.empty:
                    continue

                qv = rr["qval"].iloc[0]
                delta = rr["cliffs_delta"].iloc[0]
                label_txt = format_sig_effect(qv, delta)
                if label_txt == "":
                    continue
                if d1 not in xpos or d2 not in xpos:
                    continue

                x1, x2 = xpos[d1], xpos[d2]
                if x1 > x2:
                    x1, x2 = x2, x1

                y = base + drawn * step
                if y <= 9.8:
                    add_sig_line(ax, x1, x2, y, label_txt)
                    drawn += 1

    plt.tight_layout()
    plt.savefig(
        f"{figdir}celllevel_allclones_pairs_domains_{sample_id}.pdf",
        bbox_inches="tight",
        transparent=True
    )
    plt.show()

# Save source data + pairwise stats
all_long = []
for sample_id in samples_to_plot:
    for label in pair_sc_long_by_sample[sample_id]:
        all_long.append(pair_sc_long_by_sample[sample_id][label][0])

if len(all_long) > 0:
    pd.concat(all_long, ignore_index=True).to_csv(
        f"{outdir}fig7g_celllevel_allclones_pairs_domains_data.csv",
        index=False
    )

pairwise_stats_df.to_csv(
    f"{outdir}fig7g_celllevel_allclones_pairs_domains_pairwise_stats.csv",
    index=False
)

#### Pseudobulk level clonotype vs. domain

In [ ]:
pair_entries = [(label, vals) for label, vals in pairs_map.items() if isinstance(vals, tuple) and len(vals) == 4]
if len(pair_entries) == 0:
    raise ValueError("No pair entries (len(vals)==4) found in pairs_map.")

pair_pb_long = {}  # label -> (pb_long, tcr_type)
for label, vals in pair_entries:
    c1, c2, target_samples, tcr_type = vals

    sub_all = adata[
        (adata.obs["sample"].isin(target_samples)) &
        (adata.obs["celltype"] == "T")
    ].copy()
    if sub_all.n_obs == 0:
        continue

    clone_mask = get_clone_mask(sub_all, c1, c2)
    sub_all = sub_all[clone_mask].copy()
    if sub_all.n_obs == 0:
        continue

    sub_all.obs["domain"] = sub_all.obs["neigh_kmeans"].map(classify_3_domains)
    sub_all = sub_all[sub_all.obs["domain"].notna()].copy()

    present = sub_all.obs["domain"].value_counts()
    categories_present = [c for c in categories_order if c in present.index]
    if len(categories_present) < 2:
        continue

    X = sub_all[:, genes_to_plot].X
    if hasattr(X, "toarray"):
        X = X.toarray()

    expr_df = pd.DataFrame(X, columns=genes_to_plot, index=sub_all.obs_names)
    meta_df = sub_all.obs[["sample", "domain"]].copy()
    meta_df["domain"] = pd.Categorical(meta_df["domain"], categories=categories_present, ordered=True)

    pb = pd.concat([meta_df, expr_df], axis=1)
    pb = pb.groupby(["sample", "domain"], observed=True)[genes_to_plot].mean().reset_index()
    pb_long = pb.melt(id_vars=["sample", "domain"], var_name="gene", value_name="expr")

    pair_pb_long[label] = (pb_long, tcr_type, categories_present)

In [ ]:
pairs_map_plot = {
    'γ Clone 2': ('clone2_', ['Patient2_DX', 'Patient2_PT', 'Patient5_DX'], 'γδ'),
    'γ Clone 17': ('clone17_', ['Patient4_DX', 'Patient1_DX', 'Patient1_PT'], 'γδ'),
    'αβ Pair 1': ('clone45_', 'clone64_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 2': ('clone52_', 'clone60_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 3': ('clone51_', 'clone62_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 4': ('clone57_', 'clone87_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
}
samples_to_plot = ["Patient2_DX", "Patient2_PT", "Patient5_DX", "Patient4_DX", "Patient1_DX", "Patient1_PT"]

def get_single_clone_mask(ad, c1):
    g1 = [g for g in ad.var_names if g.startswith(c1)]
    if not g1:
        return np.zeros(ad.n_obs, dtype=bool)
    e1 = ad[:, g1].X.sum(axis=1)
    if hasattr(e1, "A1"):
        e1 = e1.A1
    else:
        e1 = np.asarray(e1).ravel()
    return e1 > 0

def get_rank_palette(d_gene, domains_plot_order):
    low_col = "#edf8b1"
    mid_col = "#7fcdbb"
    high_col = "#2c7fb8"

    centers = d_gene.groupby("domain")["expr"].mean().reindex(domains_plot_order)
    ranked = centers.dropna().sort_values().index.tolist()

    color_map = {dom: "#cccccc" for dom in domains_plot_order}
    if len(ranked) == 2:
        color_map[ranked[0]] = low_col
        color_map[ranked[1]] = high_col
    elif len(ranked) >= 3:
        color_map[ranked[0]] = low_col
        color_map[ranked[-1]] = high_col
        for dom in ranked[1:-1]:
            color_map[dom] = mid_col

    return [color_map[dom] for dom in domains_plot_order]

pair_pb_long = {}

for label, vals in pairs_map_plot.items():
    if len(vals) == 3:
        c1, allowed_samples, tcr_type = vals
        entry_type = "clone"
    elif len(vals) == 4:
        c1, c2, allowed_samples, tcr_type = vals
        entry_type = "pair"
    else:
        continue

    allowed_samples = [s for s in allowed_samples if s in samples_to_plot]

    sub_all = adata[
        (adata.obs["sample"].isin(allowed_samples)) &
        (adata.obs["celltype"] == "T")
    ].copy()
    if sub_all.n_obs == 0:
        continue

    if entry_type == "clone":
        mask = get_single_clone_mask(sub_all, c1)
    else:
        mask = get_clone_mask(sub_all, c1, c2)

    sub_all = sub_all[mask].copy()
    if sub_all.n_obs == 0:
        continue

    sub_all.obs["domain"] = sub_all.obs["neigh_kmeans"].map(classify_3_domains)
    sub_all = sub_all[sub_all.obs["domain"].notna()].copy()
    if sub_all.n_obs == 0:
        continue

    present = sub_all.obs["domain"].value_counts()
    categories_present = [c for c in categories_order if c in present.index]
    if len(categories_present) < 2:
        continue

    genes_present = [g for g in genes_to_plot if g in sub_all.var_names]
    if len(genes_present) == 0:
        continue

    X = sub_all[:, genes_present].X
    if hasattr(X, "toarray"):
        X = X.toarray()
    else:
        X = np.asarray(X)

    expr_df = pd.DataFrame(X, columns=genes_present, index=sub_all.obs_names)
    meta_df = sub_all.obs[["sample", "domain"]].copy()
    meta_df["domain"] = pd.Categorical(meta_df["domain"], categories=categories_present, ordered=True)

    pb = pd.concat([meta_df, expr_df], axis=1)
    pb = pb.groupby(["sample", "domain"], observed=True)[genes_present].mean().reset_index()
    pb_long = pb.melt(id_vars=["sample", "domain"], var_name="gene", value_name="expr")

    pair_pb_long[label] = (pb_long, tcr_type, categories_present)

plot_labels = list(pairs_map_plot.keys())
n_rows = len(plot_labels)
n_cols = len(genes_to_plot)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(14/2.54, 9/2.54),
    squeeze=False
)

for r, label in enumerate(plot_labels):
    payload = pair_pb_long.get(label, None)
    vals = pairs_map_plot[label]
    allowed_samples = vals[1] if len(vals) == 3 else vals[2]
    allowed_samples = [s for s in allowed_samples if s in samples_to_plot]

    if payload is None:
        for c in range(n_cols):
            ax = axes[r, c]
            ax.axis("off")
            if c == 0:
                ax.text(0.5, 0.5, f"{label}\n(no data)", ha="center", va="center", fontsize=FONT_SIZE)
        continue

    pb_long, tcr_type, categories_present = payload
    pb_long = pb_long[pb_long["sample"].isin(allowed_samples)].copy()

    if pb_long.empty:
        for c in range(n_cols):
            ax = axes[r, c]
            ax.axis("off")
            if c == 0:
                ax.text(0.5, 0.5, f"{label}\n(no data)", ha="center", va="center", fontsize=FONT_SIZE)
        continue

    for c, gene in enumerate(genes_to_plot):
        ax = axes[r, c]
        d_gene = pb_long[pb_long["gene"] == gene]
        if d_gene.empty:
            ax.axis("off")
            continue

        panel_palette = get_rank_palette(d_gene, categories_present)

        sns.boxplot(
            data=d_gene, x="domain", y="expr",
            order=categories_present,
            palette=panel_palette,  # <- same ranked palette logic
            showfliers=False, ax=ax,
            width=0.8, linewidth=0.5,
            boxprops={"linewidth": 0.5},
            whiskerprops={"linewidth": 0.5},
            capprops={"linewidth": 0.5},
            medianprops={"linewidth": 0.5},
        )

        ax.tick_params(axis="both", which="both", width=0.5, length=2, labelsize=FONT_SIZE)
        ax.spines["bottom"].set_visible(True)
        ax.spines["left"].set_visible(True)
        ax.spines["bottom"].set_linewidth(0.5)
        ax.spines["left"].set_linewidth(0.5)

        if r == 0:
            ax.set_title(gene, fontsize=FONT_SIZE)
        else:
            ax.set_title("")

        if r == n_rows - 1:
            ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
        else:
            ax.set_xticklabels([])
            ax.set_xlabel("")

        if c == 0:
            ax.set_ylabel(f"{label}\n({tcr_type})", fontsize=FONT_SIZE)
        else:
            ax.set_ylabel("")

        ax.grid(False)
        for side in ["top", "right"]:
            ax.spines[side].set_visible(False)

plt.tight_layout()
fig.subplots_adjust(wspace=0.8, hspace=0.1)
plt.savefig(f"{figdir}pseudobulk_all_pairs_domains.pdf", bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
comp_order = [
    ("Immune-rich", "Neuroblast-rich"),
    ("Immune-rich", "Other"),
    ("Neuroblast-rich", "Other"),
]

rows = []

for label, payload in pair_pb_long.items():
    pb_long, tcr_type, categories_present = payload
    present = set(categories_present)

    for gene in genes_to_plot:
        d_gene = pb_long[pb_long["gene"] == gene].copy()
        if d_gene.empty:
            continue

        for d1, d2 in comp_order:
            if d1 not in present or d2 not in present:
                continue

            wide = (
                d_gene[d_gene["domain"].isin([d1, d2])]
                .pivot_table(index="sample", columns="domain", values="expr", aggfunc="mean")
                .dropna(subset=[d1, d2])
            )

            n_pairs = wide.shape[0]
            if n_pairs == 0:
                continue

            diff = wide[d1] - wide[d2]
            n_nonzero = int((diff != 0).sum())

            # Effect sizes / descriptive deltas
            median_diff = float(np.median(diff))
            mean_diff = float(np.mean(diff))
            rbc_sign = np.nan if n_nonzero == 0 else float(((diff > 0).sum() - (diff < 0).sum()) / n_nonzero)

            # Optional paired test (only meaningful with >=3 paired samples)
            pval = np.nan
            valid_test = False
            if n_pairs >= 3 and n_nonzero > 0:
                try:
                    pval = float(wilcoxon(wide[d1], wide[d2], alternative="two-sided", zero_method="wilcox").pvalue)
                    valid_test = True
                except Exception:
                    pass

            rows.append({
                "pair": label,
                "tcr_type": tcr_type,
                "gene": gene,
                "domain_1": d1,
                "domain_2": d2,
                "n_paired_samples": int(n_pairs),
                "median_diff_d1_minus_d2": median_diff,
                "mean_diff_d1_minus_d2": mean_diff,
                "rank_biserial_sign": rbc_sign,  # [-1, 1], sign-based paired effect
                "pval": pval,
                "valid_test": valid_test,
            })

pb_stats_df = pd.DataFrame(rows)

# BH-FDR across all valid pseudobulk tests in this figure family
pb_stats_df["qval"] = np.nan
valid = pb_stats_df["valid_test"] & pb_stats_df["pval"].notna()
if valid.any():
    pb_stats_df.loc[valid, "qval"] = multipletests(pb_stats_df.loc[valid, "pval"], method="fdr_bh")[1]

#pb_stats_df.to_csv(f"{outdir}fig7g_pseudobulk_pairwise_stats.csv", index=False)
display(pb_stats_df.head(20))

In [ ]:
# all clones that passed the above criteria
pairs_map = {
    'γ Clone 2': ('clone2_', ['Patient2_DX', 'Patient2_PT', 'Patient5_DX'], 'γδ'),
    'γ Clone 17': ('clone17_', ['Patient4_DX', 'Patient1_DX', 'Patient1_PT'], 'γδ'),
    'αβ Pair 1': ('clone45_', 'clone64_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 2': ('clone52_', 'clone60_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 3': ('clone51_', 'clone62_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 4': ('clone57_', 'clone87_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    }

In [ ]:
pairs_map = {
    'αβ Pair 4': ('clone57_', 'clone87_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    }

In [ ]:
# Get genes to plot
genes_to_plot = [
    "CD4", "CD8A", "TRGC2",                             # lineage
    "SELL", "LEF1", "TCF7", "IL7R",                     # Naive / Tcm / TLS
    "THEMIS",                                           # TCR signal
    "MKI67",                                            # Proliferation
    "IFNG", "GZMH", "GZMB", "GZMK", "NKG7", "KLRD1",    # Cytotoxic
    "PDCD1", "LAG3", "TOX", "TIGIT"                     # Exhaustion
]

# Build sample list from pairs_map
samples_in_pairs = []
for vals in pairs_map.values():
    # vals can be:
    # (clone_prefix, [samples], type) OR (clone1, clone2, [samples], type)
    if isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
        sample_list = vals[2]
    elif isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
        sample_list = vals[1]
    else:
        continue

    for s in sample_list:
        if s not in samples_in_pairs:
            samples_in_pairs.append(s)

# Loop through each sample and plot
for sample_id in samples_in_pairs:
    print(f"\nProcessing {sample_id}...")

    # Subset to sample + T cells
    sub = adata[(adata.obs["sample"] == sample_id) & (adata.obs["celltype"] == "T")].copy()

    if sub.n_obs == 0:
        print(f"Skipping {sample_id}: No T cells found.")
        continue

    # Initialize grouping
    sub.obs["clone_group"] = "Other T cells"

    # Assign clone groups
    for label, vals in pairs_map.items():
        # vals can be:
        # (clone1, clone2, [samples], type) OR (clone_prefix, [samples], type)
        if isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
            clone_targets = [vals[0], vals[1]]
            allowed_samples = vals[2]
        elif isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
            clone_targets = [vals[0]]
            allowed_samples = vals[1]
        else:
            continue

        if sample_id not in allowed_samples:
            continue

        if len(clone_targets) == 1:
            clone_mask = get_clone_sum(sub, clone_targets[0]) > 0
        else:
            c1 = get_clone_sum(sub, clone_targets[0]) > 0
            c2 = get_clone_sum(sub, clone_targets[1]) > 0
            clone_mask = c1 & c2

        sub.obs.loc[clone_mask, "clone_group"] = label

    # Warn if no clone groups found
    if sub.obs["clone_group"].value_counts().drop("Other T cells", errors="ignore").sum() == 0:
        print(f"Warning: No specified clone groups found in {sample_id}")

    # Keep order of clones from pairs_map
    present = sub.obs["clone_group"].value_counts().index.tolist()
    order = [k for k in pairs_map.keys() if k in present] + (["Other T cells"] if "Other T cells" in present else [])

    # Add n/count to labels
    counts = sub.obs["clone_group"].value_counts().to_dict()
    label_map = {
        grp: (grp if grp == "Other T cells" else f"{grp} (n={counts.get(grp, 0)})")
        for grp in order
    }

    sub.obs["clone_group_label"] = sub.obs["clone_group"].map(label_map)
    ordered_labels = [label_map[g] for g in order]
    sub.obs["clone_group_label"] = pd.Categorical(
        sub.obs["clone_group_label"],
        categories=ordered_labels,
        ordered=True
    )

    # Validate genes exist
    current_genes = [g for g in genes_to_plot if g in sub.var_names]
    if len(current_genes) == 0:
        print(f"Skipping {sample_id}: none of genes_to_plot present.")
        continue

    dp = sc.pl.dotplot(
        sub,
        current_genes,
        groupby="clone_group_label",
        categories_order=ordered_labels,
        cmap="YlOrRd",
        return_fig=True,
        show=False,
        standard_scale="var",
    )

    dp = dp.style(
        smallest_dot=0.01,
        largest_dot=70,
    )

    # Access axes and style borders/ticks
    ax_dict = dp.get_axes()

    for _, ax in ax_dict.items():
        if ax is None:
            continue
        ax.tick_params(axis="both", which="both", width=0.5, length=2)

    main_ax = ax_dict.get("mainplot_ax", None)
    if main_ax is not None:
        for spine in main_ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_color("#000000")

    cax = ax_dict.get("color_legend_ax", None)
    if cax is not None:
        for spine in cax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_color("#000000")

    fig = dp.fig
    fig.set_size_inches(9/2.54, (len(pairs_map) + 1.5)/2.54, forward=True)
    fig.tight_layout()
    fig.canvas.draw()

    display(fig)

    fig.savefig(
        f"{figdir}dotplot_{sample_id}_supp.pdf",
        bbox_inches="tight",
        transparent=True
    )
    plt.close(fig)